# Write a Fused Softmax Kernel in Triton

**Difficulty**: 🟣 Expert

**Companies**: NVIDIA, Meta, Google, xAI, Tesla

---

### Problem Statement

Standard softmax requires **3 separate passes** over data:
1. **Pass 1**: Compute the maximum value across the row (for numerical stability)
2. **Pass 2**: Subtract the max, exponentiate each element, and accumulate the sum
3. **Pass 3**: Divide each exponentiated value by the sum

Each pass reads the entire row from global memory (HBM), making softmax **memory-bandwidth bound**. A **fused kernel** performs all three operations in a single pass by keeping data in fast on-chip memory (SRAM/registers), dramatically reducing memory traffic.

The key algorithm enabling single-pass softmax is the **online softmax** algorithm (Milakov & Gimelshein, 2018). Instead of needing the global max upfront, it maintains a **running max** and **running sum**, correcting previous partial results as a new maximum is discovered.

Your task:
1. Implement the **online softmax algorithm** in pure PyTorch (`online_softmax_pytorch`)
2. Write a **Triton fused softmax kernel** that processes one row per program instance (or provide the kernel code with a fallback if Triton is unavailable)
3. Validate both against `torch.softmax`

---

### Requirements

1. **Online Softmax** — Implement the single-pass algorithm: maintain running `max_val` and `sum_val`, updating corrections when a new max is found.
2. **Triton Kernel** — Write `fused_softmax_kernel` using `@triton.jit` that processes one row per program instance. Each program loads a row into SRAM, computes softmax in registers, and writes back.
3. **Launcher** — Write `fused_softmax(x)` that allocates output and launches the kernel with the right grid.
4. **Correctness** — All implementations must match `torch.softmax` within float32 tolerance.

---

### Constraints

- ✅ Triton kernel is optional (runs only if `triton` is importable and CUDA is available)
- ✅ The pure PyTorch online softmax must work on CPU
- ✅ Must handle arbitrary row lengths (pad to next power of 2 in the Triton kernel)
- ❌ Do **not** use `torch.softmax` in your implementation

---

<details>
  <summary>💡 Hint</summary>

  **Online softmax recurrence:**
  ```
  For each element x_i in the row:
      new_max = max(running_max, x_i)
      running_sum = running_sum * exp(running_max - new_max) + exp(x_i - new_max)
      running_max = new_max
  
  Final output: exp(x_i - final_max) / final_sum
  ```
  
  For the Triton kernel:
  - Use `tl.program_id(0)` to get the row index
  - Use `tl.arange(0, BLOCK_SIZE)` to create column offsets
  - Use a mask for columns beyond the actual row length
  - `tl.max`, `tl.exp`, and `tl.sum` operate on the loaded block

</details>

---

In [ ]:
import torch
import torch.nn.functional as F
import math

# Try to import Triton (may not be available on all systems)
TRITON_AVAILABLE = False
try:
    import triton
    import triton.language as tl
    TRITON_AVAILABLE = True
    print("Triton is available!")
except ImportError:
    print("Triton not available. Will use pure PyTorch implementation only.")

CUDA_AVAILABLE = torch.cuda.is_available()
print(f"CUDA available: {CUDA_AVAILABLE}")

In [ ]:
# Test data
torch.manual_seed(42)

# 2D input: (batch, features)
x_small = torch.randn(4, 8)
x_medium = torch.randn(32, 128)
x_large_vals = torch.tensor([[1000.0, 2000.0, 3000.0, 4000.0]])

print("x_small shape:", x_small.shape)
print("x_medium shape:", x_medium.shape)
print("x_large_vals:", x_large_vals)

In [ ]:
def online_softmax_pytorch(x: torch.Tensor) -> torch.Tensor:
    """
    Compute softmax using the online algorithm (single-pass over elements).
    Processes each row independently.
    
    The online softmax maintains a running max and running sum.
    When a new max is found, the running sum is corrected by multiplying
    by exp(old_max - new_max).
    
    Args:
        x: Input tensor of shape (M, N)
    Returns:
        Softmax output of the same shape
    """
    assert x.dim() == 2, "Input must be 2D (M, N)"
    M, N = x.shape
    output = torch.empty_like(x)
    
    for i in range(M):
        row = x[i]
        
        # TODO: Pass 1 (single pass) - compute running_max and running_sum
        # Initialize running_max to -inf and running_sum to 0
        # For each element x_j:
        #   new_max = max(running_max, x_j)
        #   running_sum = running_sum * exp(running_max - new_max) + exp(x_j - new_max)
        #   running_max = new_max
        ...
        
        # TODO: Pass 2 - compute final output using the converged max and sum
        # output[i] = exp(row - running_max) / running_sum
        ...
    
    return output

In [ ]:
# Triton kernel (runs only if Triton + CUDA available)
# If Triton is not available, this cell defines placeholder functions.

if TRITON_AVAILABLE:
    @triton.jit
    def fused_softmax_kernel(
        output_ptr, input_ptr,
        input_row_stride, output_row_stride,
        n_cols,
        BLOCK_SIZE: tl.constexpr,
    ):
        """
        Fused softmax kernel: each program instance processes one row.
        
        Steps:
        1. Load the row into SRAM (with masking for padding)
        2. Compute max for numerical stability
        3. Subtract max and exponentiate
        4. Normalize by sum
        5. Store result back to HBM
        """
        # TODO: Get the row index from program_id
        # row_idx = tl.program_id(0)
        ...
        
        # TODO: Compute pointers to the start of the input/output row
        # row_start_ptr = input_ptr + row_idx * input_row_stride
        ...
        
        # TODO: Create column offsets and mask
        # col_offsets = tl.arange(0, BLOCK_SIZE)
        # mask = col_offsets < n_cols
        ...
        
        # TODO: Load row into SRAM, using -inf for masked positions
        # row = tl.load(row_start_ptr + col_offsets, mask=mask, other=-float('inf'))
        ...
        
        # TODO: Compute softmax in SRAM
        # 1. row_max = tl.max(row, axis=0)
        # 2. numerator = tl.exp(row - row_max)
        # 3. denominator = tl.sum(numerator, axis=0)
        # 4. softmax_out = numerator / denominator
        ...
        
        # TODO: Store back to HBM
        # output_start_ptr = output_ptr + row_idx * output_row_stride
        # tl.store(output_start_ptr + col_offsets, softmax_out, mask=mask)
        ...

    def fused_softmax(x: torch.Tensor) -> torch.Tensor:
        """
        Launch the Triton fused softmax kernel.
        
        Args:
            x: Input tensor of shape (M, N) on CUDA
        Returns:
            Softmax output of the same shape
        """
        # TODO: Allocate output tensor
        # TODO: Compute BLOCK_SIZE as next power of 2 >= n_cols
        # TODO: Launch kernel with grid = (n_rows,)
        ...
else:
    print("Triton not available. Skipping kernel definition.")
    print("The fused_softmax function will not be available.")
    print("Focus on implementing online_softmax_pytorch above.")

In [ ]:
# Validation
print("=" * 60)
print("VALIDATING ONLINE SOFTMAX (PyTorch, CPU)")
print("=" * 60)

# Test 1: Small input
out_small = online_softmax_pytorch(x_small)
ref_small = torch.softmax(x_small, dim=-1)
print("\n--- Small Input Test ---")
print("Max absolute error:", (out_small - ref_small).abs().max().item())
assert torch.allclose(out_small, ref_small, atol=1e-6), "Small input test FAILED!"
print("PASSED")

# Test 2: Medium input
out_medium = online_softmax_pytorch(x_medium)
ref_medium = torch.softmax(x_medium, dim=-1)
print("\n--- Medium Input Test ---")
print("Max absolute error:", (out_medium - ref_medium).abs().max().item())
assert torch.allclose(out_medium, ref_medium, atol=1e-6), "Medium input test FAILED!"
print("PASSED")

# Test 3: Numerical stability with large values
out_large = online_softmax_pytorch(x_large_vals)
ref_large = torch.softmax(x_large_vals, dim=-1)
print("\n--- Numerical Stability Test ---")
print("Output:", out_large)
print("Reference:", ref_large)
assert not torch.isnan(out_large).any(), "NaN in output!"
assert not torch.isinf(out_large).any(), "Inf in output!"
assert torch.allclose(out_large, ref_large, atol=1e-6), "Large values test FAILED!"
print("PASSED")

# Test 4: Sum-to-1 property
print("\n--- Sum-to-1 Test ---")
sums = out_small.sum(dim=-1)
print("Row sums:", sums)
assert torch.allclose(sums, torch.ones_like(sums)), "Rows do not sum to 1!"
print("PASSED")

print("\n" + "=" * 60)
print("VALIDATING TRITON FUSED SOFTMAX (GPU)")
print("=" * 60)

if TRITON_AVAILABLE and CUDA_AVAILABLE:
    x_gpu = x_medium.cuda()
    out_triton = fused_softmax(x_gpu)
    ref_gpu = torch.softmax(x_gpu, dim=-1)
    print("\nMax absolute error:", (out_triton - ref_gpu).abs().max().item())
    assert torch.allclose(out_triton, ref_gpu, atol=1e-6), "Triton softmax FAILED!"
    print("Triton fused softmax PASSED!")
else:
    print("\nSkipping Triton validation (requires Triton + CUDA).")
    print("Online softmax PyTorch implementation validated successfully.")

print("\nAll available tests passed!")